# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [6]:
#your code here
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [8]:
spaceship.isnull().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

In [10]:
spaceship.dropna(inplace=True)

In [12]:
spaceship.isnull().sum()

PassengerId     0
HomePlanet      0
CryoSleep       0
Cabin           0
Destination     0
Age             0
VIP             0
RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
Name            0
Transported     0
dtype: int64

In [14]:
#your code here
spaceship["Cabin"].value_counts()
spaceship['Cabin'] = spaceship['Cabin'].str.split('/').str[0]
spaceship['Cabin'].unique()

array(['B', 'F', 'A', 'G', 'E', 'C', 'D', 'T'], dtype=object)

In [16]:
spaceship = spaceship.drop(['PassengerId', 'Name'], axis=1)


In [18]:
#your code here
from sklearn.datasets import fetch_openml
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

from sklearn.preprocessing import MinMaxScaler, StandardScaler

In [20]:
from sklearn.preprocessing import OneHotEncoder


In [22]:
# 1. Initialize the encoder
encoder = OneHotEncoder(drop='first', sparse_output=False)

# 2. Fit and transform the column
encoded = encoder.fit_transform(spaceship[['HomePlanet']])

# 3. Create a DataFrame from the encoded array
encoded_df = pd.DataFrame(
    encoded,
    columns=encoder.get_feature_names_out(['HomePlanet'])
)

# 4. Concatenate with original DataFrame (or drop original if needed)
spaceship = pd.concat([spaceship.drop(columns=['HomePlanet']), encoded_df], axis=1)

#print(spaceship)

**Perform Train Test Split**

In [24]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


# List of categorical columns to encode
categorical_cols = ['CryoSleep', 'Cabin', 'Destination', 'VIP']

# Drop rows with missing values in categorical columns
spaceship_cleaned = spaceship.dropna(subset=categorical_cols)

# OneHotEncoder with drop='first' and handling unknown categories
encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')

# ColumnTransformer to apply encoding only to categorical columns
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', encoder, categorical_cols)
    ],
    remainder='passthrough'  # Leave other columns (like Age) untouched
)

# Transform the data
processed = preprocessor.fit_transform(spaceship_cleaned)

# Get column names after encoding
encoded_columns = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols)
all_columns = list(encoded_columns) + [col for col in spaceship_cleaned.columns if col not in categorical_cols]

# Create final DataFrame
spaceship_encoded = pd.DataFrame(processed, columns=all_columns)

print(spaceship_encoded)

     CryoSleep_True Cabin_B Cabin_C Cabin_D Cabin_E Cabin_F Cabin_G Cabin_T  \
0               0.0     1.0     0.0     0.0     0.0     0.0     0.0     0.0   
1               0.0     0.0     0.0     0.0     0.0     1.0     0.0     0.0   
2               0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
3               0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
4               0.0     0.0     0.0     0.0     0.0     1.0     0.0     0.0   
...             ...     ...     ...     ...     ...     ...     ...     ...   
6601            0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
6602            1.0     0.0     0.0     0.0     0.0     0.0     1.0     0.0   
6603            0.0     0.0     0.0     0.0     0.0     0.0     1.0     0.0   
6604            0.0     0.0     0.0     0.0     1.0     0.0     0.0     0.0   
6605            0.0     0.0     0.0     0.0     1.0     0.0     0.0     0.0   

     Destination_PSO J318.5-22 Destination_TRAPPIST

In [26]:
spaceship_encoded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6606 entries, 0 to 6605
Data columns (total 20 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   CryoSleep_True             6606 non-null   object
 1   Cabin_B                    6606 non-null   object
 2   Cabin_C                    6606 non-null   object
 3   Cabin_D                    6606 non-null   object
 4   Cabin_E                    6606 non-null   object
 5   Cabin_F                    6606 non-null   object
 6   Cabin_G                    6606 non-null   object
 7   Cabin_T                    6606 non-null   object
 8   Destination_PSO J318.5-22  6606 non-null   object
 9   Destination_TRAPPIST-1e    6606 non-null   object
 10  VIP_True                   6606 non-null   object
 11  Age                        6606 non-null   object
 12  RoomService                6606 non-null   object
 13  FoodCourt                  6606 non-null   object
 14  Shopping

In [32]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Make a copy to avoid modifying the original
df = spaceship_encoded.copy()

# 2. Convert numeric columns to proper dtype
numeric_cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')

# 3. Drop rows with missing target
df = df.dropna(subset=['Transported'])

# 4. Define X and y
X = df.drop('Transported', axis=1)
  # Convert to binary (1/0)
y = df['Transported'].astype(int)

# 5. Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train distribution:\n", y_train.value_counts(normalize=True))


X_train shape: (5284, 19)
X_test shape: (1322, 19)
y_train distribution:
 Transported
1    0.503596
0    0.496404
Name: proportion, dtype: float64


In [30]:
print(df['Transported'].unique())


[False True]


**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [34]:
#your code here
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score

# Bagging: With replacement
bagging_clf = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,
    max_samples=0.8,
    bootstrap=True,       # Bagging = with replacement
    n_jobs=-1,
    random_state=42
)

# Pasting: Without replacement
pasting_clf = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,
    max_samples=0.8,
    bootstrap=False,      # Pasting = without replacement
    n_jobs=-1,
    random_state=42
)

# Train both models
bagging_clf.fit(X_train, y_train)
pasting_clf.fit(X_train, y_train)

# Predict on test set
y_pred_bagging = bagging_clf.predict(X_test)
y_pred_pasting = pasting_clf.predict(X_test)

# Evaluation
print("🔷 Bagging Results:")
print(classification_report(y_test, y_pred_bagging))
print(f"Accuracy: {accuracy_score(y_test, y_pred_bagging):.3f}")

print("\n🔶 Pasting Results:")
print(classification_report(y_test, y_pred_pasting))
print(f"Accuracy: {accuracy_score(y_test, y_pred_pasting):.3f}")


🔷 Bagging Results:
              precision    recall  f1-score   support

           0       0.78      0.79      0.79       656
           1       0.79      0.78      0.79       666

    accuracy                           0.79      1322
   macro avg       0.79      0.79      0.79      1322
weighted avg       0.79      0.79      0.79      1322

Accuracy: 0.788

🔶 Pasting Results:
              precision    recall  f1-score   support

           0       0.77      0.78      0.78       656
           1       0.78      0.77      0.78       666

    accuracy                           0.78      1322
   macro avg       0.78      0.78      0.78      1322
weighted avg       0.78      0.78      0.78      1322

Accuracy: 0.775


- Random Forests

In [36]:
#your code here
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# Initialize the Random Forest model
rf_clf = RandomForestClassifier(
    n_estimators=100,      # Number of trees in the forest
    max_depth=None,        # You can limit this to prevent overfitting
    random_state=42,
    n_jobs=-1              # Use all CPU cores
)

# Train the model
rf_clf.fit(X_train, y_train)

# Predict on the test set
y_pred_rf = rf_clf.predict(X_test)

# Evaluate
print("🌲 Random Forest Results:")
print(classification_report(y_test, y_pred_rf))
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.3f}")

🌲 Random Forest Results:
              precision    recall  f1-score   support

           0       0.78      0.80      0.79       656
           1       0.80      0.78      0.79       666

    accuracy                           0.79      1322
   macro avg       0.79      0.79      0.79      1322
weighted avg       0.79      0.79      0.79      1322

Accuracy: 0.790


- Gradient Boosting

In [46]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, accuracy_score

# Fix target dtype first
spaceship_encoded["Transported"] = spaceship_encoded["Transported"].astype(bool)

# Define features and target
X = spaceship_encoded.drop("Transported", axis=1)
y = spaceship_encoded["Transported"]

# Drop rows with missing y if any (just in case)
X = X[y.notna()]
y = y[y.notna()]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Impute missing values in features
imputer = SimpleImputer(strategy="most_frequent")
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

# Train Gradient Boosting Classifier
gb_clf = GradientBoostingClassifier(random_state=42)
gb_clf.fit(X_train_imputed, y_train)

# Predict and evaluate
y_pred_gb = gb_clf.predict(X_test_imputed)

print("Gradient Boosting Classifier Results:")
print(classification_report(y_test, y_pred_gb))
print(f"Accuracy: {accuracy_score(y_test, y_pred_gb):.3f}")


Gradient Boosting Classifier Results:
              precision    recall  f1-score   support

       False       0.82      0.73      0.77       656
        True       0.76      0.84      0.80       666

    accuracy                           0.79      1322
   macro avg       0.79      0.79      0.79      1322
weighted avg       0.79      0.79      0.79      1322

Accuracy: 0.787


- Adaptive Boosting

In [48]:
#your code here
from sklearn.model_selection import train_test_split
from sklearn.ensemble import AdaBoostClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, accuracy_score

# Ensure target is boolean
spaceship_encoded["Transported"] = spaceship_encoded["Transported"].astype(bool)

# Define features and target
X = spaceship_encoded.drop("Transported", axis=1)
y = spaceship_encoded["Transported"]

# Drop rows where y is missing (if any)
X = X[y.notna()]
y = y[y.notna()]

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Impute missing values in features using most frequent strategy
imputer = SimpleImputer(strategy="most_frequent")
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

# Initialize and train AdaBoost classifier
ada_clf = AdaBoostClassifier(n_estimators=100, random_state=42)
ada_clf.fit(X_train_imputed, y_train)

# Predict on test set
y_pred_ada = ada_clf.predict(X_test_imputed)

# Evaluate the classifier
print("Adaptive Boosting (AdaBoost) Results:")
print(classification_report(y_test, y_pred_ada))
print(f"Accuracy: {accuracy_score(y_test, y_pred_ada):.3f}")


/opt/anaconda3/lib/python3.12/site-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


Adaptive Boosting (AdaBoost) Results:
              precision    recall  f1-score   support

       False       0.81      0.74      0.77       656
        True       0.76      0.82      0.79       666

    accuracy                           0.78      1322
   macro avg       0.78      0.78      0.78      1322
weighted avg       0.78      0.78      0.78      1322

Accuracy: 0.781


Which model is the best and why?

In [52]:
#comment here
# I will chose Random Forest because the Accuracy: 0.790 is the highest
